# Today, I learn ONNX for the first time. We're gonna be using this a lot 

So first, what is ONNX?

ONNX = Open Neural Network Exchange

Universal translator for ML models:
- Train in PyTorch 
- Export to ONNX (universal format)
- Run on any architecture

ONNX is the bridge that takes PyTorch models to run them on the hardware

In [1]:
import torch
import torchvision

import torch.nn as nn

from torch import optim

## Step 1: Install ONNX Runtime

did this in terminal

## Step 2: Pick a trained model

For simplicity, I'll just load the Housing Regression Model I trained in week8.ipynb

In [2]:
class LinearRegression(nn.Module):

    def __init__(self,in_features):

        super().__init__()
        self.in_features = in_features

        #formula: linear --> batchnorm1d --> relu --> dropout
        self.linear_stack = nn.Sequential(
            nn.Linear(self.in_features,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.fc = nn.Linear(32,1)

    def forward(self,x):

        x = self.linear_stack(x)
        x = self.fc(x)
        return x

In [3]:
#after copying the exact model architecture, load the trained weights
model = LinearRegression(8)
model.load_state_dict(torch.load("best_housing_reg.pth", weights_only=True))
model.eval()  # IMPORTANT: must be in eval mode for export

LinearRegression(
  (linear_stack): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=32, out_features=64, bias=True)
    (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Dropout(p=0.5, inplace=False)
    (7): Linear(in_features=64, out_features=32, bias=True)
    (8): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU()
  )
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

## Step 3: Export to ONNX

In [4]:
import torch.onnx

#create a dummy input tht matches the model's expected input.
# the housing will be 1 instance with 8 dimensions
dummy = torch.randn(1,8)

#Export 
torch.onnx.export(
    model,
    dummy,
    "housing_reg.onnx",
    input_names=["input"],
    output_names = ["output"],
    dynamic_axes = {
        "input":{0:"batch_size"},
        "output":{0:"batch_size"}
    }
)

print("First ONNX export done!")

First ONNX export done!


## Step 4: Load with ONNX Runtime

In [5]:
import onnxruntime as ort 

#load ONNX model 
session = ort.InferenceSession("housing_reg.onnx")

#check input and output names
print("Input name:",session.get_inputs()[0].name)
print("Output name:",session.get_outputs()[0].name)
print("Input shape:", session.get_inputs()[0].shape)

Input name: input
Output name: output
Input shape: ['batch_size', 8]


## Step 5: Run Inference with ONNX

In [6]:
import numpy as np

#create test input (numpy array, NOT Tensor)
#in any case, it will be in numpy, just real numbers. This is 1 test row instance with 8 x's
test_input = np.random.randn(1,8).astype(np.float32)

#Run inference
result = session.run(
    ["output"],
    {"input":test_input}
)

print(f"ONNX prediction: {result[0]}")

ONNX prediction: [[2.2916145]]


## Step 6: Compare Speed - PyTorch vs. ONNX

In [8]:
import time

#Create batch of test data. 1000 test examples
test_batch = np.random.randn(1000,8).astype(np.float32)
test_tensor = torch.tensor(test_batch)

#PyTorch inference speed:
model.eval()
start = time.time()
with torch.no_grad():
    for _ in range(100):
        pytorch_output = model(test_tensor)
pytorch_time = time.time() - start
print(f"PyTorch: {pytorch_time:.4f} seconds (100 runs)")

# ONNX inference speed
start = time.time()
for _ in range(100):
    onnx_output = session.run(["output"], {"input": test_batch})
onnx_time = time.time() - start
print(f"ONNX:    {onnx_time:.4f} seconds (100 runs)")

# Compare
speedup = pytorch_time / onnx_time
print(f"\nONNX is {speedup:.2f}x faster/slower!")

PyTorch: 0.0237 seconds (100 runs)
ONNX:    0.0336 seconds (100 runs)

ONNX is 0.70x faster/slower!


## Step 7: Verify same results in PyTorch

In [9]:
# Make sure both give same predictions
with torch.no_grad():
    pytorch_pred = model(test_tensor).numpy()

onnx_pred = session.run(["output"], {"input": test_batch})[0]

# Compare
difference = np.abs(pytorch_pred - onnx_pred).mean()
print(f"Average difference: {difference:.8f}")
# Should be very close to 0 (tiny floating point differences)

Average difference: 0.00000017


### Summary and Key Takeaways

- Sessions, runtime, exports, etc. Fragments

In [ ]:
# # 1. Export PyTorch → ONNX
# dummy_input = torch.randn(1,num_feats)
# torch.onnx.export(
#     model, 
#     dummy_input, 
#     "model_name.onnx",
#     input_names=["input"],
#     output_names = ["output"],
#     dynamic_axes = {"input":{0:"batch_size"}, "output":{0:"batch_size"}}
# )

# # 2. Load ONNX model
# session = ort.InferenceSession("model.onnx")

# # 3. Run inference (numpy input, not tensor!)
# numpy_array = np.random.randn(1,8).astype(np.float32)
# result = session.run(["output"], {"input": numpy_array})

Train in PyTorch → Export to ONNX → Load InferenceSession → Run with numpy